# Analysis of the results of evaluating different embedding and retrieval models

In [1]:
import json
import pandas as pd

In [2]:
KG_DIR='/home/jmlon/Downloads/Papers/AI-KG/AI/KG'
runs = json.load(open(KG_DIR+"/reports/experiments-questions-kg-v1.json"))["runs"]
df = pd.json_normalize(runs)          # one row per run: settings.*, aggregate.*
per_question = pd.json_normalize(runs, "results", ["run_id", ["settings", "chunk_size"]])

In [3]:
df.head()

,run_id,batch_id,recorded_at,command,baseline,results,question_set.version_id,question_set.fingerprint,question_set.path,question_set.questions,...,aggregate.query_cost.p25,aggregate.query_cost.median,aggregate.query_cost.p75,aggregate.query_cost_mean,aggregate.query_cost_total,aggregate.priced_questions,aggregate.judge_cost_total,aggregate.errors,aggregate.reranked_questions,aggregate.total_tokens
0,ef55a0bc981e,d2938fb3ed2f,2026-10-31T14:10:10-05:00,eval,False,"[{'id': 'ji-2022-1', 'question': 'What data-qu...",kg-v1,b61a8db0f0e08fdb,/home/jmlon/Downloads/Papers/AI-KG/AI/KG/quest...,36,...,0.000183,0.000218,0.000241,0.000217,0.007797,36,0.017063,0,0,84333
1,a79607254703,7f813606c3a0,2026-10-31T14:10:38-05:00,eval,False,"[{'id': 'ji-2022-1', 'question': 'What data-qu...",kg-v1,b61a8db0f0e08fdb,/home/jmlon/Downloads/Papers/AI-KG/AI/KG/quest...,36,...,0.000191,0.000220,0.000272,0.000234,0.008438,36,0.017009,0,0,100739
2,8657b0656dcb,e61b5be2eee1,2026-10-31T14:11:06-05:00,eval,False,"[{'id': 'ji-2022-1', 'question': 'What data-qu...",kg-v1,b61a8db0f0e08fdb,/home/jmlon/Downloads/Papers/AI-KG/AI/KG/quest...,36,...,0.000272,0.000305,0.000371,0.000327,0.011786,36,0.016963,0,0,125806
3,9d4fc6c3c02e,d4d4a5232f0d,2026-10-31T14:11:34-05:00,eval,False,"[{'id': 'ji-2022-1', 'question': 'What data-qu...",kg-v1,b61a8db0f0e08fdb,/home/jmlon/Downloads/Papers/AI-KG/AI/KG/quest...,36,...,0.000540,0.000640,0.000694,0.000636,0.022888,36,0.016988,0,0,211381
4,4c0dfc17f869,c0d058144de7,2026-10-31T14:12:04-05:00,eval,False,"[{'id': 'ji-2022-1', 'question': 'What data-qu...",kg-v1,b61a8db0f0e08fdb,/home/jmlon/Downloads/Papers/AI-KG/AI/KG/quest...,36,...,0.000248,0.000286,0.000316,0.000282,0.010158,36,0.016697,0,36,84535


Comparison of scores and cost

In [4]:
df.groupby(["settings.llm_model", "settings.reranker"], dropna=False)[
    ["aggregate.mean_score", "aggregate.mean_coverage", "aggregate.query_cost.median"]
].mean()

aggregate.mean_score  \
settings.llm_model settings.reranker                              
openai/gpt-4o-mini voyageai/rerank-3-lite              4.175926   
                   NaN                                 3.814236   
openai/gpt-6-luna  voyageai/rerank-3-lite              4.000000   
                   NaN                                 3.750000   

                                           aggregate.mean_coverage  \
settings.llm_model settings.reranker                                 
openai/gpt-4o-mini voyageai/rerank-3-lite                 0.768750   
                   NaN                                    0.706843   
openai/gpt-6-luna  voyageai/rerank-3-lite                 0.738889   
                   NaN                                    0.687616   

                                           aggregate.query_cost.median  
settings.llm_model settings.reranker                                    
openai/gpt-4o-mini voyageai/rerank-3-lite                     0.000373  
                   NaN                                        0.000331  
openai/gpt-6-luna  voyageai/rerank-3-lite                     0.000275  
                   NaN                                        0.000171

## Last experiment: answering model and reranker (question set kg-v3)

Graded with the retrieved passages in view (`judge_protocol: reference+passages`) and priced at list price, so these runs are not comparable with kg-v1 or kg-v2. Best configuration first; at equal score, the cheaper one.

In [ ]:
runs_v3 = json.load(open(KG_DIR + "/reports/experiments-questions-kg-v3.json"))["runs"]
v3 = pd.json_normalize(runs_v3)
with_evidence = sum(q.get("passage_hit") is not None for q in runs_v3[0]["results"])

summary = pd.DataFrame({
    "model": v3["settings.llm_model"],
    "reranker": v3["settings.reranker"].fillna("—"),
    "chunk": v3["settings.chunk_size"].astype(str) + "/" + v3["settings.chunk_overlap"].astype(str),
    "top_k": v3["settings.top_k"],
    "retrieval": v3["settings.retrieval"],
    "score": v3["aggregate.mean_score"].round(2),
    "passage_hit": (v3["aggregate.passage_hit_rate"] * with_evidence).round().astype(int).astype(str)
    + f"/{with_evidence}",
    "coverage": (v3["aggregate.mean_coverage"] * 100).round().astype(int).astype(str) + "%",
    "cited": (v3["aggregate.citation_rate"] * 100).round().astype(int).astype(str) + "%",
    "usd_per_query_list": v3["aggregate.query_list_cost.median"].round(6),
    "s_per_question": v3["aggregate.question_time.median_s"].round(1),
})
# Best score first; at equal score, the cheaper configuration.
summary = summary.sort_values(["score", "usd_per_query_list"], ascending=[False, True]).reset_index(drop=True)
display(summary)

best = summary.iloc[0]
print(
    f"Best configuration: {best.score}/5, passage hit {best.passage_hit}, coverage {best.coverage}, "
    f"${best.usd_per_query_list * 1000:.2f} per 1,000 queries at list price\n"
)
print(f"""global:
    defaults:
        llm_model: {best.model}
index:
    chunk_size: {best.chunk.split('/')[0]}
    chunk_overlap: {best.chunk.split('/')[1]}
    top_k: {best.top_k}
    retrieval: {best.retrieval}
    reranker: {best.reranker if best.reranker != '—' else 'null'}""")